# Variational Circuits

This notebook covers symbolic parameters, parameter binding, sweeps, and
OpenQASM round-trips.

In [ ]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/circuit/param"
	"github.com/splch/goqu/qasm/emitter"
	"github.com/splch/goqu/qasm/parser"
	"github.com/splch/goqu/sweep"
)

In [ ]:
var (
	c     *ir.Circuit
	bound *ir.Circuit
)

var theta = param.New("theta")

## Symbolic Parameters

Create a parameter with `param.New()` and use it in symbolic rotation gates.

In [ ]:
%%
c, _ = builder.New("variational", 2).
	SymRY(theta.Expr(), 0).
	CNOT(0, 1).
	Build()
fmt.Println("Free parameters:", ir.FreeParameters(c))
gonbui.DisplaySvg(draw.SVG(c))

## Binding Parameters

`ir.Bind()` substitutes symbolic parameters with concrete values, producing a
new circuit.

In [ ]:
%%
bound, _ = ir.Bind(c, map[string]float64{"theta": math.Pi / 4})
fmt.Println("Free parameters after bind:", ir.FreeParameters(bound))
gonbui.DisplaySvg(draw.SVG(bound))

## Parameter Sweeps

Sweep a parameter across a range and collect measurement counts for each point.

In [ ]:
%%
cSweep, _ := builder.New("sweep", 2).
	SymRY(theta.Expr(), 0).
	CNOT(0, 1).
	MeasureAll().
	Build()

sw := sweep.Linspace{Key: "theta", Start: 0, Stop: math.Pi, Count: 5}
results, _ := sweep.RunSim(context.Background(), cSweep, 1000, sw)

html := "<table><tr><th>theta</th><th>|00&gt;</th><th>|11&gt;</th></tr>\n"
for _, r := range results {
	html += fmt.Sprintf("<tr><td>%.2f</td><td>%d</td><td>%d</td></tr>\n",
		r.Bindings["theta"], r.Counts["00"], r.Counts["11"])
}
html += "</table>"
gonbui.DisplayHTML(html)

## OpenQASM Round-Trip

Emit a bound circuit as OpenQASM 3.0, then parse it back.

In [ ]:
%%
qasm, _ := emitter.EmitString(bound)
fmt.Println(qasm)

In [ ]:
%%
qasm2, _ := emitter.EmitString(bound)
roundTrip, _ := parser.ParseString(qasm2)
fmt.Printf("Round-trip circuit: %s (%d qubits, depth %d)\n",
	roundTrip.Name(), roundTrip.NumQubits(), roundTrip.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(roundTrip))